# 06 — What does a BOUNDED query actually look like?

CERTAIN is easy (you get the answer) and UNACHIEVABLE is easy (you get a
refusal). This notebook shows the interesting middle: **an analyst queries
`orders AS OF day 8`, part of the evidence is destroyed, and the result
comes back BOUNDED.** What lands on their screen?

Three presentation layers, all from one query:
1. **The verdict banner** — what the tooling says before/with the result
2. **The row-level view** — observed rows and BEYOND candidates, side by side
3. **The aggregate** — `revenue ≥ $X`, a true lower bound (and when it isn't)

The data pattern matters: this table is **append-only with a retention
DELETE** (the classic lifecycle job). That's the pattern where a bounded
aggregate is a *genuine* lower bound — the destroyed rows could only have
added to the total. Section 4 shows the overwrite pattern, where BOUNDED
presents differently.

In [1]:
import shutil, time, json
from pathlib import Path
from datetime import datetime, timezone, timedelta
import pandas as pd
from deltalake import DeltaTable, write_deltalake
import alethe

WS = Path("bounded_workspace")
if WS.exists():
    shutil.rmtree(WS)
WS.mkdir()
ORDERS = WS / "orders"

# Ten days of orders, APPENDED — the log accumulates, files are per-day
for day in range(10):
    df = pd.DataFrame({
        "day":    [day] * 3,
        "order_id": [f"D{day}-{i}" for i in range(3)],
        "amount": [round(100.0 * (day + 1) + i * 10, 2) for i in range(3)],
    })
    write_deltalake(ORDERS, df, mode="append")
    time.sleep(1.0)

# The retention job: purge days 0-3, then really VACUUM
dt = DeltaTable(str(ORDERS))
dt.delete("day < 4")
removed = DeltaTable(str(ORDERS)).vacuum(
    retention_hours=0, enforce_retention_duration=False, dry_run=False)
print(f"10 days appended; retention DELETE purged days 0-3; "
      f"VACUUM destroyed {len(removed)} parquet file(s)")

10 days appended; retention DELETE purged days 0-3; VACUUM destroyed 4 parquet file(s)


In [2]:
wm = alethe.watermark(ORDERS)
print(f"chain: {wm.chain}   boundary: v{wm.boundary['version']}   "
      f"validated: {wm.empirically_validated}")

# The analyst's target: 'AS OF day 8' — the commit time of day 8's append,
# which is BEFORE the boundary. This is a BOUNDED query.
def commit_time(version: int) -> datetime:
    log = ORDERS / "_delta_log" / f"{version:020d}.json"
    for line in log.read_text().splitlines():
        a = json.loads(line)
        if "commitInfo" in a:
            return datetime.fromtimestamp(a["commitInfo"]["timestamp"] / 1000,
                                          tz=timezone.utc)

as_of = commit_time(8)
v = alethe.verdict(wm, since=as_of)
print(f"\nquery: SELECT day, SUM(amount) FROM orders AS OF {as_of.strftime('%H:%M:%S')}")
print(f"verdict: {v}")

chain: delta://orders   boundary: v10   validated: True

query: SELECT day, SUM(amount) FROM orders AS OF 23:32:44
verdict: Verdict(BOUNDED, limiting=['delta://orders'])


## 1. What the naive engine does with this query

No verdict, no warning — the log happily resolves the timestamp, and the
read dies.

In [3]:
try:
    dt = DeltaTable(str(ORDERS))
    dt.load_as_version(as_of)
    dt.to_pyarrow_table()
except FileNotFoundError as e:
    print(f"AS OF {as_of.strftime('%H:%M:%S')} → FileNotFoundError")
    print(f"  {str(e).splitlines()[0][:100]}")
    print("\nThe analyst gets a stack trace — or on some engines, silently")
    print("fewer rows. Neither says 'part of your answer was destroyed.'")

AS OF 23:32:44 → FileNotFoundError
  Object at location /Users/seamusaran/Documents/alethe/notebooks/bounded_workspace/orders/part-00000-

The analyst gets a stack trace — or on some engines, silently
fewer rows. Neither says 'part of your answer was destroyed.'


## 2. The BOUNDED presentation — layer by layer

### Layer 1: the verdict banner

This is what tooling shows *with* the result — the epistemic header.

In [4]:
report = alethe.pit_report("orders", [wm])
zone = report.query(as_of)

print("┌" + "─" * 68 + "┐")
print(f"│ ⚠ BOUNDED RESULT{' ' * 51}│")
print(f"│ Retention has destroyed part of {zone.limiting_chains[0]:<27}       │")
print(f"│ before {wm.boundary_dt.strftime('%H:%M:%S')} (boundary v{wm.boundary['version']})."
      f" Rows below are the SURVIVING evidence. │")
print(f"│ SUM/COUNT over non-negative measures are LOWER BOUNDS.{' ' * 12}│")
print("└" + "─" * 68 + "┘")

┌────────────────────────────────────────────────────────────────────┐
│ ⚠ BOUNDED RESULT                                                   │
│ Retention has destroyed part of delta://orders                    │
│ before 23:32:46 (boundary v10). Rows below are the SURVIVING evidence. │
│ SUM/COUNT over non-negative measures are LOWER BOUNDS.            │
└────────────────────────────────────────────────────────────────────┘


### Layer 2: the row-level view

The result table itself. Two kinds of rows:

- **OBSERVED** — surviving facts, values as recorded
- **BEYOND** — *candidates*: row-shapes we know existed (the calendar says
  days 0–3 happened; the manifest says their evidence was destroyed), with
  unknowable values. They are **in the result**, not silently dropped —
  that's what stops them vanishing from joins and group-bys.

Because this table is append-only, every surviving row with `day ≤ 8` is a
true fact about day 8 — appends after day 8 have later day keys and are
filtered out; the destroyed rows could only have *added* days 0–3.

In [5]:
# The surviving evidence: current state (readable), filtered to day <= 8
surviving = (DeltaTable(str(ORDERS)).to_pyarrow_table().to_pandas()
             .query("day <= 8")
             .groupby("day", as_index=False)
             .agg(n_orders=("order_id", "size"), revenue=("amount", "sum")))
surviving["epistemic"] = "OBSERVED"

# The destroyed population: known row-shapes (the calendar), unknowable values
destroyed = pd.DataFrame({
    "day": range(0, 4),
    "n_orders": [pd.NA] * 4,
    "revenue": [pd.NA] * 4,
    "epistemic": ["BEYOND"] * 4,
})

result = pd.concat([destroyed, surviving], ignore_index=True).astype(object)
result = result.where(result.notna(), "— unknowable —")
print(f"orders AS OF {as_of.strftime('%H:%M:%S')} — revenue by day:\n")
print(result.to_string(index=False))

orders AS OF 23:32:44 — revenue by day:

day       n_orders        revenue epistemic
  0 — unknowable — — unknowable —    BEYOND
  1 — unknowable — — unknowable —    BEYOND
  2 — unknowable — — unknowable —    BEYOND
  3 — unknowable — — unknowable —    BEYOND
  4              3         1530.0  OBSERVED
  5              3         1830.0  OBSERVED
  6              3         2130.0  OBSERVED
  7              3         2430.0  OBSERVED
  8              3         2730.0  OBSERVED


### Layer 3: the aggregate — a true lower bound

The analyst asked for one number. BOUNDED means they get a **floor**, with
the unknowable contribution named — not a fake exact number, and not a
refusal of a question that has a partially-known answer.

In [6]:
floor = surviving.revenue.sum()
n_beyond = len(destroyed)
print(f"SELECT SUM(amount) FROM orders AS OF day-8\n")
print(f"  revenue ≥ ${floor:,.2f}")
print(f"  (BOUNDED: {n_beyond} day(s) of evidence destroyed by retention —")
print(f"   their non-negative contribution is unknowable; the true total")
print(f"   can only be HIGHER. Limiting chain: {zone.limiting_chains[0]})")
print()
print("Compare the three possible presentations of the same query:")
print(f"  naive engine     : FileNotFoundError (or silently ${floor:,.2f} as 'exact')")
print(f"  refuse everything: ERROR — no answer")
print(f"  BOUNDED          : ≥ ${floor:,.2f}, gaps named, floor guaranteed")

SELECT SUM(amount) FROM orders AS OF day-8

  revenue ≥ $10,650.00
  (BOUNDED: 4 day(s) of evidence destroyed by retention —
   their non-negative contribution is unknowable; the true total
   can only be HIGHER. Limiting chain: delta://orders)

Compare the three possible presentations of the same query:
  naive engine     : FileNotFoundError (or silently $10,650.00 as 'exact')
  refuse everything: ERROR — no answer
  BOUNDED          : ≥ $10,650.00, gaps named, floor guaranteed


## 3. The same mechanics, by algebra

The row-level presentation above isn't ad hoc — it's the observability
semiring. Load the same data into a `TemporalTable` with the retention
watermark and query it: a query past the boundary emits every known
row-shape as a BEYOND *candidate*, the taint flows through joins by `⊗`,
survives projection by `⊕`, and `split_result` separates answers from
refusals with zero special-case code.

In [7]:
from alethe import TemporalTable, split_result, K

# Day-granularity temporal table: history before day 4 is vacuumed
t_orders = TemporalTable(
    name="orders_daily",
    columns=("day", "revenue"),
    key_columns=("day",),
    retention_watermark=4,          # days < 4 destroyed
)
for row in surviving.itertuples():
    t_orders.insert_version((row.day, float(row.revenue)), valid_from=row.day)
for day in range(4):                # the destroyed days: known keys only
    t_orders.insert_version((day, None), valid_from=day)

# Query INSIDE retention: clean, every row OBSERVED
r_ok = split_result(t_orders.as_of(8))
print(f"as_of(day 8): refused={r_ok.refused}, "
      f"{len(r_ok.answers.data)} OBSERVED rows — CERTAIN presentation\n")

# Query BEFORE the watermark: every known row-shape returns as a candidate
r_bounded = split_result(t_orders.as_of(2))
print(f"as_of(day 2): refused={r_bounded.refused}")
print(f"  answers  : {len(r_bounded.answers.data)}")
print(f"  refusals : {len(r_bounded.refusals.data)} candidate rows, "
      f"all annotated {K.BEYOND!r}")
print("\n  The refusals are ROWS, not an error — they keep flowing through")
print("  joins (⊗ taints), survive projection (⊕), and arrive in the result")
print("  where the presentation layer renders them as '— unknowable —'.")

as_of(day 8): refused=False, 9 OBSERVED rows — CERTAIN presentation

as_of(day 2): refused=True
  answers  : 0
  refusals : 9 candidate rows, all annotated BEYOND

  The refusals are ROWS, not an error — they keep flowing through
  joins (⊗ taints), survive projection (⊕), and arrive in the result
  where the presentation layer renders them as '— unknowable —'.


## 4. When BOUNDED is *not* a lower bound — the overwrite pattern

Everything above relied on append-only semantics: surviving rows ⊆ true
rows, so the sum is a floor. An **overwrite-mode** table (each load
replaces the whole state — most snapshot-style dims and facts) breaks that
subset property: reading the boundary state gives you a *different day's*
rows, not *fewer of the right day's* rows.

In [8]:
OVR = WS / "balances"   # overwrite-mode: daily account balances
for v, total in enumerate([500.0, 800.0, 300.0, 650.0]):   # note: goes DOWN too
    write_deltalake(OVR, pd.DataFrame({"account": ["A"], "balance": [total],
                                       "as_of_day": [v]}), mode="overwrite")
    time.sleep(1.0)
DeltaTable(str(OVR)).vacuum(retention_hours=0,
                            enforce_retention_duration=False, dry_run=False)
wm_b = alethe.watermark(OVR)
bv = wm_b.boundary["version"]

clamped = DeltaTable(str(OVR), version=bv).to_pyarrow_table().to_pandas()
print(f"query: balance AS OF day 1   (true answer was $800 — destroyed)")
print(f"clamped read at boundary v{bv}: ${clamped.balance.iloc[0]:,.2f}\n")
print("┌" + "─" * 68 + "┐")
print("│ ⚠ BOUNDED (temporal substitution)                                  │")
print(f"│ This is the state at the boundary (day {int(clamped.as_of_day.iloc[0])}),"
      " the earliest evidence   │")
print("│ that survives — NOT the state at day 1, and NOT a lower bound:    │")
print("│ balances move both ways. Value is real; its time is wrong.        │")
print("└" + "─" * 68 + "┘")

query: balance AS OF day 1   (true answer was $800 — destroyed)
clamped read at boundary v3: $650.00

┌────────────────────────────────────────────────────────────────────┐
│ ⚠ BOUNDED (temporal substitution)                                  │
│ This is the state at the boundary (day 3), the earliest evidence   │
│ that survives — NOT the state at day 1, and NOT a lower bound:    │
│ balances move both ways. Value is real; its time is wrong.        │
└────────────────────────────────────────────────────────────────────┘


## Recap — how BOUNDED presents

| Layer | Presentation |
|---|---|
| Verdict banner | `⚠ BOUNDED — retention destroyed part of delta://orders before <boundary>; limiting chain named` |
| Row level | OBSERVED rows with values, **plus** BEYOND candidate rows rendered `— unknowable —`, still present so joins/aggregations can't silently lose them |
| Aggregate (append-only) | `revenue ≥ $X` — a guaranteed floor, gaps named |
| Aggregate (overwrite) | boundary state, stamped **temporal substitution** — real value, wrong time, explicitly *not* a bound |

The common thread: **BOUNDED is an answer, not an error** — but one that
carries its own epistemic status everywhere it goes.